# 03 — Clean Minimum Wage Data

This notebook processes the FRED state minimum wage history into a clean annual panel that can be
merged with the QCEW employment data in the border-county causal inference analysis.

**Input:** `data/raw/min_wage/state_mw_history.csv`
**Output:** `data/intermediate/min_wage_panel.parquet`

In [1]:
import pandas as pd
from pathlib import Path

# ---------------------------------------------------------------------------
# Paths
# ---------------------------------------------------------------------------
ROOT = Path().resolve().parent
RAW_MIN_WAGE = ROOT / "data" / "raw" / "min_wage"
INTERMEDIATE = ROOT / "data" / "intermediate"
INTERMEDIATE.mkdir(parents=True, exist_ok=True)
OUTPUT = INTERMEDIATE / "min_wage_panel.parquet"

## 1. Inspect the raw CSV

In [2]:
# Read raw file — keep everything as-is for inspection
raw = pd.read_csv(RAW_MIN_WAGE / "state_mw_history.csv")

print("Shape  :", raw.shape)
print("Dtypes :")
print(raw.dtypes)
print("\nHead:")
raw.head(10)

Shape  : (7796, 3)
Dtypes :
state           str
date            str
min_wage    float64
dtype: object

Head:


,state,date,min_wage
0,AL,1938-10-01,0.25
1,AL,1938-11-01,0.25
2,AL,1938-12-01,0.25
3,AL,1939-01-01,0.25
4,AL,1939-02-01,0.25
5,AL,1939-03-01,0.25
6,AL,1939-04-01,0.25
7,AL,1939-05-01,0.25
8,AL,1939-06-01,0.25
9,AL,1939-07-01,0.25


In [3]:
# Basic descriptive checks on the raw data
print("Unique states :", raw["state"].nunique())
print("States        :", sorted(raw["state"].unique().tolist()))
print("\nDate range    :", raw["date"].min(), "→", raw["date"].max())
print("Min wage range:", raw["min_wage"].min(), "→", raw["min_wage"].max())
print("\nMissing values:")
print(raw.isnull().sum())

Unique states : 51
States        : ['AK', 'AL', 'AR', 'AZ', 'CA', 'CO', 'CT', 'DC', 'DE', 'FL', 'GA', 'HI', 'IA', 'ID', 'IL', 'IN', 'KS', 'KY', 'LA', 'MA', 'MD', 'ME', 'MI', 'MN', 'MO', 'MS', 'MT', 'NC', 'ND', 'NE', 'NH', 'NJ', 'NM', 'NV', 'NY', 'OH', 'OK', 'OR', 'PA', 'RI', 'SC', 'SD', 'TN', 'TX', 'UT', 'VA', 'VT', 'WA', 'WI', 'WV', 'WY']

Date range    : 1938-10-01 → 2026-03-01
Min wage range: 0.25 → 17.95

Missing values:
state       0
date        0
min_wage    0
dtype: int64


## 2. Check monthly structure — multiple entries per state per year

In [4]:
# Parse date so we can extract year and month
raw["date"] = pd.to_datetime(raw["date"])
raw["year"] = raw["date"].dt.year
raw["month"] = raw["date"].dt.month

# How many observations per state per year?
obs_per_state_year = (
    raw.groupby(["state", "year"])["min_wage"].count().rename("n_obs").reset_index()
)

print("Observations per state-year (should be ~12 for monthly FRED series):")
print(obs_per_state_year["n_obs"].value_counts().sort_index())

# Flag any state-years with fewer than 12 observations (may affect edge years)
short = obs_per_state_year[obs_per_state_year["n_obs"] < 12]
if short.empty:
    print("\nAll state-years have 12 observations — clean monthly series.")
else:
    print(f"\nState-years with < 12 observations ({len(short)} cases):")
    print(short.head(20))

Observations per state-year (should be ~12 for monthly FRED series):
n_obs
1     2546
3       10
12     435
Name: count, dtype: int64

State-years with < 12 observations (2556 cases):
   state  year  n_obs
0     AK  1968      1
1     AK  1969      1
2     AK  1970      1
3     AK  1971      1
4     AK  1972      1
5     AK  1973      1
6     AK  1974      1
7     AK  1975      1
8     AK  1976      1
9     AK  1977      1
10    AK  1978      1
11    AK  1979      1
12    AK  1980      1
13    AK  1981      1
14    AK  1982      1
15    AK  1983      1
16    AK  1984      1
17    AK  1985      1
18    AK  1986      1
19    AK  1987      1


## 3. Filter to 2003–2024 and take January values

In [5]:
# Keep only the analysis window (2003 = pre-period baseline, 2024 = last available year)
FIRST_YEAR = 2003
LAST_YEAR = 2024

df = raw[(raw["year"] >= FIRST_YEAR) & (raw["year"] <= LAST_YEAR)].copy()
print(f"Rows after year filter ({FIRST_YEAR}–{LAST_YEAR}): {len(df):,}")

# Retain only the January observation as the annual minimum wage.
# January captures the value in force at the start of the year — the standard
# approach in the literature (Dube, Lester & Reich 2010; Neumark & Wascher 2008).
df_jan = df[df["month"] == 1].copy()
print(f"Rows after keeping January only : {len(df_jan):,}")
print(
    f"Expected (51 states x {LAST_YEAR - FIRST_YEAR + 1} years) : {51 * (LAST_YEAR - FIRST_YEAR + 1):,}"
)

Rows after year filter (2003–2024): 2,323
Rows after keeping January only : 1,113
Expected (51 states x 22 years) : 1,122


## 4. Cast types and clean up columns

In [6]:
# Cast min_wage to float
df_jan["min_wage"] = df_jan["min_wage"].astype(float)

# Cast year to int
df_jan["year"] = df_jan["year"].astype(int)

# Drop helper columns we no longer need
df_jan = df_jan.drop(columns=["date", "month"])

print("Dtypes after cleaning:")
print(df_jan.dtypes)
df_jan.head(10)

Dtypes after cleaning:
state           str
min_wage    float64
year          int64
dtype: object


,state,min_wage,year
771,AL,5.15,2003
783,AL,5.15,2004
795,AL,5.15,2005
807,AL,5.15,2006
819,AL,5.15,2007
831,AL,5.85,2008
843,AL,6.55,2009
855,AL,7.25,2010
867,AL,7.25,2011
879,AL,7.25,2012


## 5. Add state FIPS codes

In [7]:
# Load the Census Bureau state FIPS table
state_fips_df = pd.read_csv(
    ROOT / "data" / "raw" / "state_fips.txt",
    sep="|",
    dtype=str,
)[["STATE", "STUSAB"]].rename(columns={"STATE": "state_fips", "STUSAB": "state"})

# Zero-pad FIPS to 2 digits
state_fips_df["state_fips"] = state_fips_df["state_fips"].str.zfill(2)

print(f"Loaded {len(state_fips_df)} states/territories from Census FIPS table")
print(state_fips_df.head())

# Merge FIPS codes onto the panel
df_jan = df_jan.merge(state_fips_df, on="state", how="left")

# Verify — any states that failed to match?
unmatched = df_jan[df_jan["state_fips"].isna()]["state"].unique()
if len(unmatched) == 0:
    print("\nAll states matched to FIPS codes.")
else:
    print(f"\nWARNING: {len(unmatched)} states did not match: {unmatched}")

Loaded 57 states/territories from Census FIPS table
  state_fips state
0         01    AL
1         02    AK
2         04    AZ
3         05    AR
4         06    CA

All states matched to FIPS codes.


## 6. Finalise panel and inspect

In [8]:
# Select and order final columns
panel = df_jan[["state", "state_fips", "year", "min_wage"]].copy()

# Sort for readability
panel = panel.sort_values(["state", "year"]).reset_index(drop=True)

print("=== Final panel ===")
print("Shape  :", panel.shape)
print("Dtypes :")
print(panel.dtypes)
print(f"\nStates : {panel['state'].nunique()} unique")
print(f"Years  : {sorted(panel['year'].unique().tolist())}")
print()
panel.head(20)

=== Final panel ===
Shape  : (1113, 4)
Dtypes :
state             str
state_fips        str
year            int64
min_wage      float64
dtype: object

States : 51 unique
Years  : [2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]



,state,state_fips,year,min_wage
0,AK,02,2003,7.15
1,AK,02,2004,7.15
2,AK,02,2005,7.15
3,AK,02,2006,7.15
4,AK,02,2007,7.15
5,AK,02,2008,7.15
6,AK,02,2009,7.15
7,AK,02,2010,7.75
8,AK,02,2011,7.75
9,AK,02,2012,7.75


In [9]:
# Quick summary statistics for min_wage
print("Min wage summary statistics:")
print(panel["min_wage"].describe())

# Check for low or high values
print("\nLowest min wages:")
print(
    panel.nsmallest(10, "min_wage")[["state", "year", "min_wage"]].to_string(
        index=False
    )
)

print("\nHighest min wages (spot-check):")
print(
    panel.nlargest(10, "min_wage")[["state", "year", "min_wage"]].to_string(index=False)
)

Min wage summary statistics:
count    1113.000000
mean        7.759475
std         2.235495
min         2.650000
25%         6.750000
50%         7.250000
75%         8.250000
max        17.500000
Name: min_wage, dtype: float64

Lowest min wages:
state  year  min_wage
   KS  2003      2.65
   KS  2004      2.65
   KS  2005      2.65
   KS  2006      2.65
   KS  2007      2.65
   KS  2008      2.65
   KS  2009      2.65
   NM  2003      4.25
   OH  2003      4.25
   OH  2004      4.25

Highest min wages (spot-check):
state  year  min_wage
   DC  2024     17.50
   DC  2023     16.50
   WA  2024     16.28
   DC  2022     16.10
   CA  2024     16.00
   NY  2024     16.00
   OR  2024     15.95
   WA  2023     15.74
   CT  2024     15.69
   CA  2023     15.50


In [10]:
# Confirm panel is balanced: each state should appear once per year
counts = panel.groupby("state")["year"].count()
expected_years = LAST_YEAR - FIRST_YEAR + 1

if (counts == expected_years).all():
    print(
        f"Panel is balanced: every state has exactly {expected_years} observations ({FIRST_YEAR}–{LAST_YEAR})."
    )
else:
    print("Panel is unbalanced — some states have unexpected row counts:")
    print(counts[counts != expected_years])

Panel is unbalanced — some states have unexpected row counts:
state
AZ    18
FL    19
GA    21
WY    21
Name: year, dtype: int64


In [11]:
full_index = pd.MultiIndex.from_product(
    [panel["state"].unique(), range(FIRST_YEAR, LAST_YEAR + 1)], names=["state", "year"]
)
missing = full_index.difference(pd.MultiIndex.from_frame(panel[["state", "year"]]))
print(
    pd.DataFrame(missing.tolist(), columns=["state", "year"]).sort_values(
        ["state", "year"]
    )
)

  state  year
0    AZ  2003
1    AZ  2004
2    AZ  2005
3    AZ  2006
4    FL  2003
5    FL  2004
6    FL  2005
7    GA  2024
8    WY  2024



***Data Note: Missing State Minimum Wage Observations***

Four states have fewer observations than the full 2003–2024 range:

- **Arizona (2003–2006):** Arizona had no state minimum wage law prior to 2007. 
  The state adopted its own minimum wage through Proposition 202, which took 
  effect January 1, 2007. Years before this are excluded rather than imputed 
  to avoid misrepresenting the policy environment.

- **Florida (2003–2004):** Florida's minimum wage amendment passed in November 
  2004 and took effect in May 2005. The two years prior to the law are excluded 
  for the same reason.

- **Georgia (2024) and Wyoming (2024):** Both states maintain a statutory minimum 
  wage of $5.15, which falls below the federal floor of $7.25. FRED does not 
  report a 2024 value for either state. 
  
  In practice, workers in both states are covered by the federal 
  minimum, but since the analysis uses state-level statutory rates and no valid 
  value exists, exclusion is the more defensible choice.

All four cases are dropped from the panel rather than imputed to preserve the 
integrity of the policy variation used for identification.

In [12]:
drop_conditions = [
    ("AZ", [2003, 2004, 2005, 2006]),
    ("FL", [2003, 2004, 2005]),
    ("GA", [2024]),
    ("WY", [2024]),
]

for state, years in drop_conditions:
    panel = panel[~((panel["state"] == state) & (panel["year"].isin(years)))]

panel = panel.sort_values(["state", "year"]).reset_index(drop=True)

## 7. Save to parquet and verify

In [13]:
# Save as Parquet
panel.to_parquet(OUTPUT, index=False)